# NYC Building Energy Benchmarking - Part A1: Data Profiling
## TY B.Tech Integrated Data Warehouse, OLAP and Data Mining Project
**Domain:** Building Energy and Sustainability Analytics (NYC Local Law 84)

### Objectives:
1. Ingest raw benchmarking disclosures from `data/raw/nyc_building_energy.csv`
2. Profile missing values count and percentage for all attributes
3. Identify exact vs logical duplicate records
4. Profile numeric ranges (min, max, mean, median, zero/negative counts, IQR outliers)
5. Verify categorical consistency and codes
6. Generate structured profiling report in `data/staging/profiling_report.json`


In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
RAW_PATH = '../data/raw/nyc_building_energy.csv'
REPORT_PATH = '../data/staging/profiling_report.json'
os.makedirs('../data/staging', exist_ok=True)

print(f"Loading raw dataset from: {RAW_PATH}")
df_raw = pd.read_csv(RAW_PATH)
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()


Loading raw dataset from: ../data/raw/nyc_building_energy.csv
Raw Dataset Shape: 7,515 rows × 13 columns


### 1. Column Overview and Missing Value Profiling

In [2]:
null_counts = df_raw.isnull().sum()
null_pcts = (null_counts / len(df_raw)) * 100
dtypes = df_raw.dtypes

profile_df = pd.DataFrame({
    'Column Name': df_raw.columns,
    'Data Type': dtypes.values,
    'Null Count': null_counts.values,
    'Null Percentage (%)': np.round(null_pcts.values, 2),
    'Distinct Values': [df_raw[c].nunique() for c in df_raw.columns]
})

print("=== Attribute Quality and Completeness Summary ===")
profile_df


=== Attribute Quality and Completeness Summary ===


### 2. Duplicate Analysis (Exact vs Logical Duplicates)

In [3]:
exact_dups = int(df_raw.duplicated().sum())

pid_col = [c for c in df_raw.columns if 'property_id' in c.lower() or 'propertyid' in c.lower()][0]
yr_col = [c for c in df_raw.columns if 'year' in c.lower()][0]

logical_dups = int(df_raw.duplicated(subset=[pid_col, yr_col]).sum())

print(f"Exact Duplicate Records: {exact_dups}")
print(f"Logical Duplicate Records ({pid_col}, {yr_col}): {logical_dups}")


Exact Duplicate Records: 15
Logical Duplicate Records (property_id, reporting_year): 15


### 3. Numeric Attribute Range and Outlier Analysis

In [4]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns

num_summary = []
for col in numeric_cols:
    s = df_raw[col].dropna()
    q25 = s.quantile(0.25)
    q75 = s.quantile(0.75)
    iqr = q75 - q25
    outliers = int(((s < (q25 - 1.5 * iqr)) | (s > (q75 + 1.5 * iqr))).sum())
    zeros = int((s == 0).sum())
    negs = int((s < 0).sum())
    
    num_summary.append({
        'Attribute': col,
        'Min': round(float(s.min()), 2),
        'Q25': round(float(q25), 2),
        'Median': round(float(s.median()), 2),
        'Mean': round(float(s.mean()), 2),
        'Q75': round(float(q75), 2),
        'Max': round(float(s.max()), 2),
        'Zeros': zeros,
        'Negatives': negs,
        'IQR Outliers': outliers
    })

pd.DataFrame(num_summary)


### 4. Categorical Code Validation

In [5]:
borough_col = [c for c in df_raw.columns if 'borough' in c.lower()][0]
print(f"Distinct Boroughs: {df_raw[borough_col].unique()}")

type_col = [c for c in df_raw.columns if 'type' in c.lower()][0]
print(f"Distinct Property Types count: {df_raw[type_col].nunique()}")
df_raw[type_col].value_counts().head(10)


Distinct Boroughs: ['Brooklyn' 'Manhattan' 'Queens' 'Bronx' 'BX' 'Staten Island' 'MN' 'BK'
 'QN']
Distinct Property Types count: 12


### 5. Save Data Profiling Report

In [6]:
profiling_result = {
    'total_rows': int(len(df_raw)),
    'total_columns': int(len(df_raw.columns)),
    'exact_duplicates': exact_dups,
    'logical_duplicates': logical_dups,
    'columns': profile_df.astype(str).to_dict(orient='records'),
    'numeric_profiles': num_summary
}

with open(REPORT_PATH, 'w') as f:
    json.dump(profiling_result, f, indent=2)

print(f"Profiling report successfully saved to: {REPORT_PATH}")


Profiling report successfully saved to: ../data/staging/profiling_report.json
